# Cruces entre datasets (caso 1.3)

El caso 1.3 pide **agrupar los distritos de Madrid según su perfil de riesgo y su contexto social**, cruzando el socioeconómico con las activaciones (por ejemplo, ver si la renta media se relaciona con ciertos tipos de emergencia, o si la densidad influye en los tiempos de llegada). Bluetab nos pidió además mirar qué otros datasets se pueden cruzar para sacar más información.

Aquí revisamos dataset a dataset qué cruces son posibles, cuáles usamos y por qué, y montamos la tabla final: **una fila por distrito**, que es la que entra en el clustering.

**Orden:** se ejecuta después de `limpieza_socioeconomico.ipynb`, `limpieza_distritos.ipynb` y `limpieza_activaciones.ipynb`, porque usa los CSV limpios que generan.

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

def cargar(nombre):
    # Los CSV limpios están junto al notebook; los catálogos originales en ../data.
    ruta = Path(nombre)
    if not ruta.exists():
        ruta = Path('../data') / nombre
    return pd.read_csv(ruta)

def normalizar_texto(valor):
    if pd.isna(valor):
        return None
    s = unicodedata.normalize('NFD', str(valor).strip().upper())
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')

soc = cargar('socioeconomico_limpio.csv')
dis = cargar('distritos_limpio.csv')
act = cargar('activaciones_limpio.csv')

print(f"socioeconomico: {soc.shape}, distritos: {dis.shape}, activaciones: {act.shape}")

socioeconomico: (21, 15), distritos: (21, 7), activaciones: (152107, 12)


## 1. Socioeconómico + distritos: la base

Las dos tablas tienen una fila por distrito y el mismo `distrito_id`, así que se unen directo y encajan las 21 de 21. Es la tabla maestra: de aquí salen población, superficie y densidad, coherentes con los indicadores sociales.

In [2]:
# Quitamos del socioeconomico las columnas que ya estan en distritos (poblacion y superficie),
# porque son las mismas (se comprobo en la limpieza) y si no saldrian duplicadas.
base = dis.merge(soc.drop(columns=['poblacion_2024', 'superficie_km2']), on='distrito_id')
print(f"Filas tras unir: {len(base)} (de 21)")
print(f"Distritos sin pareja: {base['distrito'].isna().sum()}")
base[['distrito_id', 'nombre', 'poblacion', 'superficie_km2', 'renta_media_hogar',
      'indice_vulnerabilidad', 'densidad_hab_km2']].head()

Filas tras unir: 21 (de 21)
Distritos sin pareja: 0


,distrito_id,nombre,poblacion,superficie_km2,renta_media_hogar,indice_vulnerabilidad,densidad_hab_km2
0,1,CENTRO,149718,5.23,42831.0,0.35,28626.8
1,2,ARGANZUELA,155115,6.47,39876.0,0.42,23974.5
2,3,RETIRO,119915,5.46,NaN,0.22,21962.5
3,4,SALAMANCA,147707,5.41,71432.0,0.15,27302.6
4,5,CHAMARTIN,146512,9.19,62145.0,0.18,15942.5


## 2. Activaciones por distrito: la demanda

Las activaciones ya traen el distrito normalizado de la limpieza. Quedan fuera del análisis por distrito las 876 filas con distrito `NO CONSTA` (0,6%) y 201 filas de fuera del término municipal (`FUERA DE TERMINO` y `CARRETERAS Y CIRCUNVALACIONES`). Siguen en el CSV limpio y cuentan en la demanda total de Madrid, pero no pertenecen a ningún distrito de la ciudad.

Agrupamos por distrito y calculamos:

- **Nº de activaciones** y **activaciones por 1.000 habitantes**. Dividimos por población porque si no, los distritos grandes siempre parecen peores solo por tener más gente.
- **% de prioridad alta**, como indicador de gravedad de la demanda.
- **Edad media de los pacientes**, que se puede comparar con la edad media del distrito.

El tiempo de respuesta todavía no se usa: hay activaciones con la intervención antes que la solicitud que pueden ser cruces de medianoche, y la regla para tratarlas está pendiente de validar con Bluetab. Cuando se valide, se añadirá aquí el tiempo medio por distrito.

In [3]:
# Solo los 21 distritos de Madrid ciudad: fuera NO CONSTA y las activaciones de fuera del termino
act_distrito = act[act['distrito'].isin(set(dis['nombre']))].copy()
print(f"Activaciones con distrito asignado: {len(act_distrito)} de {len(act)} "
      f"(quedan fuera {len(act) - len(act_distrito)}, el "
      f"{(len(act) - len(act_distrito)) / len(act) * 100:.2f}%)")

demanda = act_distrito.groupby('distrito').agg(
    n_activaciones=('activacion_id', 'count'),
    pct_prioridad_alta=('prioridad', lambda s: (s == 'ALTA').mean() * 100),
    edad_media_paciente=('paciente_edad', 'mean'),
).round(2).reset_index()

print(f"Distritos con activaciones: {len(demanda)}")
demanda.head()

Activaciones con distrito asignado: 151030 de 152107 (quedan fuera 1077, el 0.71%)
Distritos con activaciones: 21


,distrito,n_activaciones,pct_prioridad_alta,edad_media_paciente
0,ARGANZUELA,8104,29.73,49.29
1,BARAJAS,2385,30.36,50.65
2,CARABANCHEL,10642,29.73,49.40
3,CENTRO,22144,29.79,50.03
4,CHAMARTIN,7260,30.29,48.98


## 3. Qué aportan los demás datasets

Bluetab nos dijo que mirásemos otros datasets para cruzar información. Esto es lo que hemos encontrado, dataset a dataset.

In [4]:
uni = cargar('unidades_samur.csv')
bases = cargar('bases_samur.csv')
hosp = cargar('hospitales.csv')
tipos = cargar('tipos_emergencia.csv')
acc = cargar('accidentes.csv')

# UNIDADES SAMUR: encaja el 100% por unidad_id -> carga de trabajo por unidad y por base.
match_uni = act['unidad_id'].isin(set(uni['unidad_id'])).mean() * 100
print(f"1. unidades_samur: encaja el {match_uni:.0f}% por unidad_id.")

# TIPOS DE EMERGENCIA: encaja el 92%, pero hay codigos repetidos con categoria distinta.
tipos['cod_norm'] = tipos['codigo'].map(normalizar_texto)
dups = tipos[tipos.duplicated('cod_norm', keep=False)].sort_values('cod_norm')
match_tip = act['tipo_emergencia'].map(normalizar_texto).isin(set(tipos['cod_norm'])).mean() * 100
afectadas = act['tipo_emergencia'].map(normalizar_texto).isin(set(dups['cod_norm'])).sum()
print(f"2. tipos_emergencia: encaja el {match_tip:.0f}%, pero {len(dups)//2} codigos salen dos veces "
      f"con categoria y prioridad distintas ({afectadas} activaciones afectadas):")
print(dups[['codigo', 'categoria', 'prioridad_base']].to_string(index=False))

# BASES Y HOSPITALES: catalogos por distrito -> oferta sanitaria.
print(f"3. bases_samur: {len(bases)} bases; hospitales: {len(hosp)} hospitales. "
      f"Se cruzan por distrito (normalizando el nombre).")

# ACCIDENTES: trafico, con municipios de fuera de Madrid y distritos que faltan.
en_catalogo = acc['distrito'].map(normalizar_texto).isin(set(dis['nombre'])).sum()
print(f"4. accidentes: solo {en_catalogo} de {len(acc)} filas tienen un distrito de Madrid ciudad; "
      f"el resto son municipios de fuera o nombres ambiguos.")

# METEOROLOGIA Y CALENDARIO: van por dia y activaciones no tiene dia.
print("5. meteorologia y calendario: van por dia. Activaciones solo tiene año, mes y hora, "
      "asi que no se pueden cruzar por fecha.")

1. unidades_samur: encaja el 100% por unidad_id.


2. tipos_emergencia: encaja el 92%, pero 2 codigos salen dos veces con categoria y prioridad distintas (14223 activaciones afectadas):
              codigo    categoria prioridad_base
               Caída       Médica           ALTA
               CAIDA       Trauma          Media
Intoxicación etílica Psiquiátrica           ALTA
INTOXICACION ETILICA Intoxicación          media
3. bases_samur: 15 bases; hospitales: 20 hospitales. Se cruzan por distrito (normalizando el nombre).
4. accidentes: solo 1596 de 2062 filas tienen un distrito de Madrid ciudad; el resto son municipios de fuera o nombres ambiguos.
5. meteorologia y calendario: van por dia. Activaciones solo tiene año, mes y hora, asi que no se pueden cruzar por fecha.


### Lo que decidimos con cada uno

- **unidades_samur: sí.** Encaja el 100% por `unidad_id` (unir por id, no por el código, que se repite). Lo usamos para la oferta: cuántas unidades tiene cada distrito en su base.
- **bases_samur y hospitales: sí.** Se cruzan por distrito y dan la oferta sanitaria: bases, unidades, hospitales y camas. El caso pregunta justo por la relación entre densidad y recursos del SAMUR.
- **tipos_emergencia: todavía no.** `INTOXICACION ETILICA` y `CAIDA` salen dos veces en el catálogo con categoría y prioridad distintas, y entre las dos suman 14.223 activaciones. Hasta que Bluetab diga cuál manda, no unimos el catálogo; para los tipos más frecuentes usamos el texto de la propia activación.
- **accidentes: no para el clustering.** Es de tráfico, incluye municipios de fuera de Madrid (Alcorcón, Getafe, Móstoles...) que no están en el socioeconómico, y le faltan distritos. Se queda como posible ampliación (riesgo vial por distrito).
- **meteorología y calendario: no se puede.** Van por día y las activaciones no tienen día (no existe en la fuente y ya se lo hemos trasladado a Bluetab). Cruzar solo por año-mes sería una asociación demasiado gruesa para el clustering.

## 4. Tabla final: una fila por distrito

Juntamos la base (socioeconómico + distritos) con la demanda (activaciones agrupadas) y la oferta (bases, unidades, hospitales y camas). El resultado es `tabla_distritos.csv`: 21 filas, una por distrito, con el perfil social, la demanda y los recursos. Es la tabla que entra en el clustering.

In [5]:
# Oferta por distrito: bases, unidades y hospitales
bases['distrito_norm'] = bases['distrito'].map(normalizar_texto)
hosp['distrito_norm'] = hosp['distrito'].map(normalizar_texto)
uni['distrito_norm'] = uni['base_distrito'].map(normalizar_texto)

oferta_bases = bases.groupby('distrito_norm').agg(num_bases_samur=('base_id', 'count')).reset_index()
oferta_unidades = uni.groupby('distrito_norm').agg(unidades_samur=('unidad_id', 'count')).reset_index()
oferta_hosp = hosp.groupby('distrito_norm').agg(
    num_hospitales_cat=('hospital_id', 'count'), camas_hospital=('camas', lambda s: s.sum(min_count=1))).reset_index()

tabla = (base
         .merge(demanda, left_on='nombre', right_on='distrito', how='left')
         .merge(oferta_bases, left_on='nombre', right_on='distrito_norm', how='left')
         .merge(oferta_unidades, left_on='nombre', right_on='distrito_norm', how='left')
         .merge(oferta_hosp, left_on='nombre', right_on='distrito_norm', how='left'))

tabla['activaciones_por_1000hab'] = (tabla['n_activaciones'] / tabla['poblacion'] * 1000).round(2)

# Una base sin hospitales en el catalogo no es un cero inventado: el catalogo puede no cubrirlo todo.
# Lo marcamos vacio en vez de 0 para no afirmar algo que no sabemos.
cols_tabla = ['distrito_id', 'nombre', 'poblacion', 'superficie_km2', 'densidad_hab_km2',
              'renta_media_hogar', 'edad_media', 'pct_mayores_65', 'pct_extranjeros',
              'tasa_paro', 'indice_vulnerabilidad', 'num_centros_salud', 'parques_m2_habitante',
              'n_activaciones', 'activaciones_por_1000hab', 'pct_prioridad_alta',
              'edad_media_paciente', 'num_bases_samur', 'unidades_samur',
              'num_hospitales_cat', 'camas_hospital']
tabla = tabla[cols_tabla].rename(columns={'nombre': 'distrito'})
tabla.to_csv('tabla_distritos.csv', index=False)
print(tabla.shape)   # (21, 21)
tabla.head(10)

(21, 21)


,distrito_id,distrito,poblacion,superficie_km2,densidad_hab_km2,renta_media_hogar,edad_media,pct_mayores_65,pct_extranjeros,tasa_paro,...,num_centros_salud,parques_m2_habitante,n_activaciones,activaciones_por_1000hab,pct_prioridad_alta,edad_media_paciente,num_bases_samur,unidades_samur,num_hospitales_cat,camas_hospital
0,1,CENTRO,149718,5.23,28626.8,42831.0,44.2,22.1,21.5,8.2,...,8.0,3.2,22144,147.90,29.79,50.03,1.0,3.0,NaN,NaN
1,2,ARGANZUELA,155115,6.47,23974.5,39876.0,42.8,18.5,18.2,9.5,...,6.0,8.5,8104,52.25,29.73,49.29,1.0,NaN,1.0,NaN
2,3,RETIRO,119915,5.46,21962.5,NaN,47.3,25.3,12.3,6.8,...,5.0,25.6,5380,44.87,29.29,49.12,NaN,NaN,1.0,1500.0
3,4,SALAMANCA,147707,5.41,27302.6,71432.0,48.1,26.8,11.5,5.2,...,6.0,4.8,7705,52.16,29.84,49.31,NaN,NaN,1.0,480.0
4,5,CHAMARTIN,146512,9.19,15942.5,62145.0,46.5,NaN,10.8,5.8,...,7.0,6.2,7260,49.55,30.29,48.98,1.0,NaN,1.0,150.0
5,6,TETUAN,155159,5.37,28893.7,35678.0,41.2,17.8,22.4,11.2,...,6.0,2.8,8106,52.24,30.25,49.86,1.0,11.0,NaN,NaN
6,7,CHAMBERI,137532,4.68,29387.2,51234.0,45.8,23.5,13.5,7.5,...,8.0,3.5,7123,51.79,29.33,49.46,NaN,NaN,NaN,NaN
7,8,FUENCARRAL-EL PARDO,245974,237.84,1034.2,48765.0,42.1,16.2,12.1,8.1,...,12.0,45.2,6302,25.62,29.69,49.54,1.0,NaN,1.0,1328.0
8,9,MONCLOA-ARAVACA,123606,46.29,2670.3,55432.0,43.5,19.8,11.2,NaN,...,6.0,18.5,8422,68.14,30.04,49.04,1.0,NaN,2.0,1574.0
9,10,LATINA,233765,25.37,9214.2,31245.0,43.8,20.5,18.5,12.5,...,10.0,5.8,8625,36.90,30.02,49.53,NaN,11.0,NaN,NaN


## 5. Primer vistazo: renta, vulnerabilidad y demanda

El caso pregunta si los barrios más vulnerables tienen una demanda distinta. Como primer vistazo (sin sacar conclusiones todavía, que eso es el clustering), miramos las correlaciones entre la demanda por 1.000 habitantes y los indicadores sociales, y los tipos de emergencia más frecuentes.

In [6]:
cols_corr = ['activaciones_por_1000hab', 'renta_media_hogar', 'indice_vulnerabilidad',
             'pct_mayores_65', 'tasa_paro', 'densidad_hab_km2', 'edad_media_paciente']
print("Correlaciones con la demanda por 1.000 habitantes:")
print(tabla[cols_corr].corr()['activaciones_por_1000hab'].round(2).sort_values())
print()
print("Tipos de emergencia mas frecuentes (top 10):")
print(act['tipo_emergencia'].value_counts().head(10).to_string())
print()
tabla.describe().round(2)

Correlaciones con la demanda por 1.000 habitantes:
indice_vulnerabilidad      -0.27
tasa_paro                  -0.26
edad_media_paciente         0.21
renta_media_hogar           0.22
pct_mayores_65              0.33
densidad_hab_km2            0.44
activaciones_por_1000hab    1.00
Name: activaciones_por_1000hab, dtype: float64

Tipos de emergencia mas frecuentes (top 10):
tipo_emergencia
PATOLOGIA CARDIOVASCULAR         28144
CASUAL: CAIDA, ETC               26013
INTOXICACION ETILICA             14036
AGRESION SIN ESPECIFICAR          8611
ACCIDENTE MENOS DE 3 VICTIMAS     7877
HERIDAS                           6690
PATOLOGIA PSICOSOMATICA           5732
CERTIFICACION PSIQUIATRICO        5359
OTROS                             5188
ACCIDENTE DE MOTOCICLETA          4675



,distrito_id,poblacion,superficie_km2,densidad_hab_km2,renta_media_hogar,edad_media,pct_mayores_65,pct_extranjeros,tasa_paro,indice_vulnerabilidad,num_centros_salud,parques_m2_habitante,n_activaciones,activaciones_por_1000hab,pct_prioridad_alta,edad_media_paciente,num_bases_samur,unidades_samur,num_hospitales_cat,camas_hospital
count,21.0,21.00,21.00,21.00,19.00,21.00,19.00,21.00,18.00,19.00,20.00,21.00,21.00,21.00,21.00,21.00,14.0,6.00,9.00,8.00
mean,11.0,157604.19,28.80,14410.11,40614.16,42.95,18.59,16.55,10.56,0.46,7.00,11.97,7191.90,47.04,29.67,49.48,1.0,6.50,1.11,985.00
std,6.2,56809.83,50.08,9945.07,13429.27,2.74,4.14,4.58,3.68,0.20,2.51,11.51,4115.19,25.45,0.58,0.45,0.0,3.56,0.33,534.81
min,1.0,47703.00,4.68,1034.20,23456.00,37.20,11.20,10.50,5.20,0.15,3.00,2.80,2385.00,25.62,27.73,48.63,1.0,3.00,1.00,150.00
25%,6.0,123606.00,6.31,6838.00,30110.50,41.50,16.00,12.30,7.65,0.30,5.75,4.50,5068.00,34.81,29.52,49.24,1.0,4.25,1.00,495.00
50%,11.0,149718.00,14.05,15942.50,35678.00,42.80,18.20,16.80,10.00,0.45,6.50,6.50,6844.00,41.36,29.73,49.46,1.0,5.00,1.00,1174.00
75%,16.0,193104.00,28.24,21962.50,49999.50,44.20,21.30,20.80,12.73,0.56,8.25,15.80,8106.00,51.79,30.02,49.67,1.0,9.50,1.00,1371.00
max,21.0,257329.00,237.84,29387.20,71432.00,48.10,26.80,25.60,17.20,0.82,12.00,45.20,22144.00,147.90,30.36,50.65,1.0,11.00,2.00,1574.00


## Pendiente (preguntas para Bluetab)

- **Regla de medianoche:** para el tiempo de respuesta, proponemos sumar 24 h solo cuando el cruce de medianoche sea plausible (intervención poco después de las 00:00 y solicitud poco antes) y dejar nulo el resto, conservando la fila. Pendiente de validar; cuando se valide se añade el tiempo medio por distrito a la tabla.
- **Día de la activación:** no existe en la fuente (solo año, mes y horas), así que meteorología y calendario no se pueden cruzar. Ya se lo hemos trasladado por si tienen la fecha completa.
- **tipos_emergencia duplicados:** `INTOXICACION ETILICA` y `CAIDA` aparecen dos veces con categoría y prioridad distintas. ¿Cuál es la versión correcta?
- **Huecos del socioeconómico:** renta de Retiro y Ciudad Lineal, % de mayores de 65 de Chamartín y Ciudad Lineal, paro de Moncloa-Aravaca, Hortaleza y Villa de Vallecas, vulnerabilidad de Carabanchel y Barajas, centros de salud de San Blas-Canillejas. ¿Hay fuente oficial para rellenarlos?
- **Índice de vulnerabilidad:** hemos asumido que va de 0 a 1 (el 1,5 de Carabanchel pasó a vacío). ¿Es correcto?
- **Población de Barajas:** estimada como densidad × superficie (47.703). ¿Tienen el dato real?